In [103]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import warnings
import json
import joblib
import shutil
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from tensorflow.keras.models import Sequential, load_model as _lm
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, f1_score
from scipy.stats import randint as sp_randint
from scipy.optimize import minimize

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [104]:
df = pd.read_csv('data/Calorie Prediction/exercise_dataset.csv')
print('Columns in Dataset', df.columns.tolist())
print(f'Shape of Dataset: {df.shape[0]:,} rows x {df.shape[1]} cols')

Columns in Dataset ['ID', 'Exercise', 'Calories Burn', 'Dream Weight', 'Actual Weight', 'Age', 'Gender', 'Duration', 'Heart Rate', 'BMI', 'Weather Conditions', 'Exercise Intensity']
Shape of Dataset: 3,864 rows x 12 cols


In [105]:
print('Exercise and Fitness Metrics Dataset')
display(df.head(3))

Exercise and Fitness Metrics Dataset


,ID,Exercise,Calories Burn,Dream Weight,Actual Weight,Age,Gender,Duration,Heart Rate,BMI,Weather Conditions,Exercise Intensity
0,1,Exercise 2,286.959851,91.892531,96.301115,45,Male,37,170,29.426275,Rainy,5
1,2,Exercise 7,343.453036,64.165097,61.104668,25,Male,43,142,21.286346,Rainy,5
2,3,Exercise 4,261.223465,70.846224,71.766724,20,Male,20,148,27.899592,Cloudy,4


In [106]:
print('Calories Burn stats:')
print(df['Calories Burn'].describe().round(1))
print()

Calories Burn stats:
count    3864.0
mean      301.9
std       115.8
min       100.0
25%       202.2
50%       299.7
75%       404.1
max       499.9
Name: Calories Burn, dtype: float64



In [107]:
print('Dataset Info:')
print(df.info())
print()

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 3864 entries, 0 to 3863
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  3864 non-null   int64  
 1   Exercise            3864 non-null   str    
 2   Calories Burn       3864 non-null   float64
 3   Dream Weight        3864 non-null   float64
 4   Actual Weight       3864 non-null   float64
 5   Age                 3864 non-null   int64  
 6   Gender              3864 non-null   str    
 7   Duration            3864 non-null   int64  
 8   Heart Rate          3864 non-null   int64  
 9   BMI                 3864 non-null   float64
 10  Weather Conditions  3864 non-null   str    
 11  Exercise Intensity  3864 non-null   int64  
dtypes: float64(4), int64(5), str(3)
memory usage: 439.6 KB
None



In [108]:
dfClean = df.copy()
dfClean.drop(columns=['ID', 'Exercise', 'Dream Weight'], inplace=True)

dfClean.rename(columns={
    'Calories Burn':'Calories_Burned','Actual Weight':'Weight','Heart Rate':'HR','Weather Conditions':'Weather_Conditions','Exercise Intensity':'Exercise_Intensity','Duration':'Exercise_Duration'
}, inplace=True)

# Replacing Calories Burn Feature with Real Values
Original Calories Burn in the Dataset in Synthetic, Because Random Noise and Uniform data between 100 to 500 calories with near zero corr with every other fearture So we are replacing it with a validated calories formula (but we are using the existing columns in the dataset to get the calories burned)

In [109]:
#Adding these featuers to see the correlations with Calories_Burned Before the fix
print('Correlations with Calories_Burned After Fix:')
for col in ['Exercise_Duration', 'HR', 'Exercise_Intensity', 'Weight', 'Age']:
    correlation = dfClean[col].corr(dfClean["Calories_Burned"])
    print(f'    {col}:  {correlation:+.4f}')

Correlations with Calories_Burned After Fix:
    Exercise_Duration:  +0.0218
    HR:  -0.0359
    Exercise_Intensity:  +0.0108
    Weight:  +0.0104
    Age:  -0.0011


In [110]:
# Formula used:(Keytel et al., 2005,standard in fitness ML literature)

durMin = dfClean['Exercise_Duration'].values
weight = dfClean['Weight'].values
age = dfClean['Age'].values
hr = dfClean['HR'].values
intensity = dfClean['Exercise_Intensity'].values
gender = dfClean['Gender'].values

weather_factor = dfClean['Weather_Conditions'].map({
    'Sunny' : 1.05,
    'Cloudy': 1.00,
    'Rainy' : 1.08
})
male_cal   = durMin * (-55.0969 + 0.6309*hr + 0.1988*weight + 0.2017*age) / 4.184
female_cal = durMin * (-20.4022 + 0.4472*hr - 0.05741*weight + 0.074*age) / 4.184

cal_base   = np.where(gender == 'Male', male_cal, female_cal)

intensity_factor = 0.85 + (intensity - 1) * 0.03 #Adjust for exercise intensity scale (±15% range around baseline)

noise = np.random.normal(0, 18, len(dfClean)) # Add realistic measurement noise (±18 cal std dev)

dfClean['Calories_Burned'] = np.clip(
    cal_base * intensity_factor * weather_factor + noise,
    80, 700
)

In [111]:
dfClean['Exercise_Duration'] = dfClean['Exercise_Duration'] / 60 #Converting to hours after formula

In [112]:
if dfClean.isnull().sum().sum()  == 0:
    print("No Null Values")
else:
    print('Null Values Available')

print(f'Columns in Dataset: {dfClean.columns.tolist()}')

No Null Values
Columns in Dataset: ['Calories_Burned', 'Weight', 'Age', 'Gender', 'Exercise_Duration', 'HR', 'BMI', 'Weather_Conditions', 'Exercise_Intensity']


In [113]:
print(dfClean['Calories_Burned'].describe().round(1))

count    3864.0
mean      452.6
std       170.7
min        80.0
25%       312.1
50%       442.1
75%       606.2
max       700.0
Name: Calories_Burned, dtype: float64


In [114]:
x = dfClean['Calories_Burned'].skew()

x_val = float(x)
if abs(x_val) < 0.05:
    print(f'Skew: {x_val} (near zero value so no log transform needed)')
else:
    print(f'Skew: {x_val} (consider log transform)')


Skew: 0.017390949724929033 (near zero value so no log transform needed)


In [115]:
print('Correlations with Calories_Burned After Fix:')
for col in ['Exercise_Duration', 'HR', 'Exercise_Intensity', 'Weight', 'Age']:
    correlation = dfClean[col].corr(dfClean["Calories_Burned"])
    print(f'    {col}:  {correlation:+.4f}')

Correlations with Calories_Burned After Fix:
    Exercise_Duration:  +0.6719
    HR:  +0.5920
    Exercise_Intensity:  +0.1816
    Weight:  +0.0370
    Age:  +0.0577


In [116]:
def removeOutliers(df, col, k=3.0):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - k*IQR, Q3 + k*IQR
    before = len(df)
    df_clean = df[(df[col] >= lo) & (df[col] <= hi)]
    print(f'  {col}: Removed {before - len(df_clean)} outliers '
          f'(IQR range [{lo:.0f}, {hi:.0f}])')
    return df_clean

print('Outlier removal DS1:')
dfClean = removeOutliers(dfClean, 'Calories_Burned')

Outlier removal DS1:
  Calories_Burned: Removed 0 outliers (IQR range [-570, 1489])


In [117]:
print('Missing Values:')
Missing = dfClean.isnull().sum()
print(Missing[Missing > 0].to_string())
print()

print('Missing Value Percentage:')
MissingPerc = dfClean.isnull().mean() * 100
print(MissingPerc[MissingPerc > 0].to_string())

Missing Values:
Series([], )

Missing Value Percentage:
Series([], )


In [118]:
dfClean.to_csv('data/Cleaned/cleaned_data.csv', index=False)

In [119]:
#Using binary encoding because variable has only two states, and a single binary feature already encodes all the information with no loss.
dfClean['Gender'] = dfClean['Gender'].map({'Male': 1, 'Female': 0})
assert dfClean['Gender'].isnull().sum() == 0

#One-hot encoding for Weather_Conditions since it has multiple categories and no ordinal relationship.
dfClean = pd.get_dummies(dfClean, columns=['Weather_Conditions'], drop_first=True)

# Convert boolean columns to integers (0 and 1) for modeling compatibility.
bool_cols = dfClean.select_dtypes(include='bool').columns
dfClean[bool_cols] = dfClean[bool_cols].astype(int)

In [120]:
print(f'Encoded. Shape: {dfClean.shape}')

Encoded. Shape: (3864, 10)


In [121]:
before = len(dfClean)
dfClean.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(dfClean)}')
print(f'Final cleaned shape: {dfClean.shape}')

Duplicates removed: 0
Final cleaned shape: (3864, 10)


In [122]:
dfClean['Max_HR_Percentage'] = dfClean['HR'] / (220 - dfClean['Age'].clip(1, 100))
dfClean['Workload'] = dfClean['Exercise_Intensity'] * dfClean['Exercise_Duration']
dfClean['Age_Group'] = pd.cut(dfClean['Age'],bins=[0, 25, 45, 65, 100],labels=[0, 1, 2, 3]).astype(int)

In [123]:
dfClean.dtypes

Calories_Burned             float64
Weight                      float64
Age                           int64
Gender                        int64
Exercise_Duration           float64
HR                            int64
BMI                         float64
Exercise_Intensity            int64
Weather_Conditions_Rainy      int64
Weather_Conditions_Sunny      int64
Max_HR_Percentage           float64
Workload                    float64
Age_Group                     int64
dtype: object